# Classificador IPC — mesma receita da 1ª rodada sobre embeddings Qwen3-Embedding-8B (TAC)

Reproduz o treino do classificador IPC da amostra PoC 1 (pedidos de 2026, `patentes_xml_embeddings_ipc.csv`)
trocando só os vetores: **Gemma 2 TAC** (média TA + Claims, 768 dim.) → **Qwen3-Embedding-8B TAC** (texto
concatenado título + resumo + reivindicações, 4.096 dim.).

Tudo o mais é fixo e vem do pacote `poc1_*`:
- mesma divisão treino / validação / teste (13.565 / 2.928 / 2.918 pedidos, divisão agrupada, semente 42);
- mesmo universo de rótulos (2.585 grupos principais com suporte ≥ 2, 575 subclasses);
- mesma receita: logística um-contra-todos, Adam (lr 0,02), lotes de 1.024, 30 épocas, sem penalidade;
- mesma decodificação hierárquica: nota do grupo = log σ(grupo) + log σ(subclasse-mãe);
- referência: Gemma 2 TAC com essa receita (grupo Hit@1 0,532 no teste), com acertos por pedido para
  o teste pareado.

Cada entrada de `VETORES` é uma pasta, ou um par (pasta TA, pasta C) para usar a média dos dois vetores.
Os resultados trazem também os pedidos **longos** (texto TAC acima de 4.096 tokens; 330 no teste) e **curtos**.

Entradas: `poc1_rotulos.csv.gz`, `poc1_universo.json` e a pasta dos vetores (`embeddings_*.npy` + `pedidos*.csv`,
em arquivo único ou em blocos). Roda em GPU se houver; em CPU também funciona (mais lento).

In [ ]:
# Configuração
from pathlib import Path
PACOTE = Path(".")                                    # pasta com poc1_rotulos.csv.gz e poc1_universo.json
VETORES = {"qwen3_8b_tac": Path("8B/TAC")}            # nome -> pasta com embeddings_*.npy e pedidos*.csv
# média de dois vetores, como o "TAC" do Gemma 2: nome -> (pasta TA, pasta C), p. ex.
# VETORES = {"qwen3_8b_media_ta_c": (Path("8B/TA"), Path("8B/C"))}
LIMITE_LONGOS = 4096                                  # pedidos "longos": texto TAC acima disso (330 no teste)
SAIDA = Path("resultados_classificador_qwen8b")
EPOCAS, LAM, LOTE, LR, SEMENTE = 30, 0.0, 1024, 0.02, 0   # receita da 1ª rodada (não alterar para comparar)
RARO = 10                                             # grupo raro: menos de 10 pedidos no treino
N_BOOTSTRAP = 2000

In [ ]:
import csv, glob, gzip, json, re, time
import numpy as np
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
SAIDA.mkdir(parents=True, exist_ok=True)

universo = json.load(open(PACOTE / "poc1_universo.json", encoding="utf-8"))
GRUPOS, SUBCLASSES = universo["grupos"], universo["subclasses"]
gpos = {c: i for i, c in enumerate(GRUPOS)}; spos = {c: i for i, c in enumerate(SUBCLASSES)}
PAI = np.array([spos[g[:4]] for g in GRUPOS])          # subclasse-mãe de cada grupo

linhas = list(csv.DictReader(gzip.open(PACOTE / "poc1_rotulos.csv.gz", "rt", encoding="utf-8")))
IDS = [r["numero_pedido"] for r in linhas]
PARTE = np.array([r["split"] for r in linhas])
TOKENS = np.array([int(r["tokens_tac_qwen3"]) for r in linhas])

def rotulos(ipc):
    codigos = {c.strip().replace(" ", "") for c in ipc.split(";") if c.strip()}
    return {c.split("/")[0] + "/00" for c in codigos}, {c[:4] for c in codigos}

YG = np.zeros((len(linhas), len(GRUPOS)), bool); YS = np.zeros((len(linhas), len(SUBCLASSES)), bool)
for i, r in enumerate(linhas):
    g, s = rotulos(r["ipc"])
    for x in g & gpos.keys(): YG[i, gpos[x]] = True
    for x in s & spos.keys(): YS[i, spos[x]] = True
IDX = {p: np.where(PARTE == p)[0] for p in ("train", "validation", "test")}
print({p: len(v) for p, v in IDX.items()}, "| grupos", len(GRUPOS), "| subclasses", len(SUBCLASSES))
assert {p: len(v) for p, v in IDX.items()} == {"train": 13565, "validation": 2928, "test": 2918}

In [ ]:
def carregar_vetores(pasta):
    """Lê embeddings_*.npy + pedidos*.csv (arquivo único ou blocos) e alinha à ordem de poc1_rotulos."""
    pasta = Path(pasta)
    embs = sorted(pasta.glob("embeddings*.npy"))
    assert embs, f"nenhum embeddings*.npy em {pasta}"
    vetores, ids = [], []
    for e in embs:
        p = pasta / e.name.replace("embeddings", "pedidos").replace(".npy", ".csv")
        if not p.exists():
            candidatos = sorted(pasta.glob("pedidos*.csv"))
            assert len(embs) == 1 and len(candidatos) == 1, f"sem pedidos correspondente a {e.name}"
            p = candidatos[0]
        v = np.load(e)
        lidos = [r["numero_pedido"].strip() for r in csv.DictReader(open(p, encoding="utf-8-sig"))]
        assert len(lidos) == len(v), (e.name, len(lidos), len(v))
        vetores.append(v); ids += lidos
    V = np.vstack(vetores)
    pos = {n: i for i, n in enumerate(ids)}
    faltam = [n for n in IDS if n not in pos]
    assert not faltam, f"{len(faltam)} pedidos sem vetor, ex.: {faltam[:5]}"
    X = V[[pos[n] for n in IDS]].astype(np.float32)
    X /= np.linalg.norm(X, axis=1, keepdims=True)
    meta = {}
    for nome in ("metadata.json", "config.json", "resultado.json"):
        if (pasta / nome).exists():
            meta = json.load(open(pasta / nome, encoding="utf-8")); break
    return X, meta

In [ ]:
def treinar(X, Y):
    """Logística um-contra-todos com Adam: tradução para PyTorch de hierarquia_piloto.train_flat
    (mesma inicialização, mesmos lotes pela mesma semente, mesmos hiperparâmetros)."""
    X = torch.as_tensor(X, device=DEVICE); Y = torch.as_tensor(Y, dtype=torch.float32, device=DEVICE)
    K, dim = Y.shape[1], X.shape[1]
    W = torch.zeros((K, dim), device=DEVICE)
    prior = Y.mean(0).clamp(1e-4, 1 - 1e-4); c = torch.log(prior / (1 - prior))
    mW, vW, mc, vc = torch.zeros_like(W), torch.zeros_like(W), torch.zeros_like(c), torch.zeros_like(c)
    rng = np.random.default_rng(SEMENTE); n = len(X); t = 0
    for _ in range(EPOCAS):
        for idx in np.array_split(rng.permutation(n), max(1, n // LOTE)):
            t += 1
            ii = torch.as_tensor(idx, device=DEVICE); xb = X[ii]
            err = (torch.sigmoid(xb @ W.T + c) - Y[ii]) / len(idx)
            for p_, g_, m_, v_ in ((W, err.T @ xb + LAM * W, mW, vW), (c, err.sum(0), mc, vc)):
                m_.mul_(0.9).add_(0.1 * g_); v_.mul_(0.999).add_(0.001 * g_ * g_)
                p_.sub_(LR * (m_ / (1 - 0.9 ** t)) / (torch.sqrt(v_ / (1 - 0.999 ** t)) + 1e-8))
    return W.cpu().numpy(), c.cpu().numpy()

def log_sigmoide(z):
    return -np.logaddexp(0.0, -z)

FILHOS = [np.where(PAI == k)[0] for k in range(len(SUBCLASSES))]
SUPORTE_TREINO = YG[IDX["train"]].sum(0)

def decodificar(X, Wg, bg, Ws, bs):
    """Nota hierárquica dos grupos e ordem das subclasses pelo seu melhor grupo (igual à API)."""
    nota = log_sigmoide(X @ Wg.T + bg) + log_sigmoide(X @ Ws.T + bs)[:, PAI]
    melhor = np.full((len(X), len(SUBCLASSES)), -np.inf)
    for k, f in enumerate(FILHOS):
        if len(f): melhor[:, k] = nota[:, f].max(1)
    return np.argsort(-nota, 1)[:, :5], np.argsort(-melhor, 1)[:, :3]

def metricas(og, os_, linhas_idx):
    yg, ys = YG[linhas_idx], YS[linhas_idx]
    h = np.take_along_axis(yg, og, 1); hs = np.take_along_axis(ys, os_, 1)
    r = {f"grupo_hit@{k}": float(h[:, :k].any(1).mean()) for k in (1, 3, 5)}
    r.update({f"subclasse_hit@{k}": float(hs[:, :k].any(1).mean()) for k in (1, 3)})
    em5 = np.zeros_like(yg); np.put_along_axis(em5, og, True, 1)
    for nome, m in (("raros", SUPORTE_TREINO < RARO), ("frequentes", SUPORTE_TREINO >= RARO)):
        r[f"recall@5_{nome}"] = float((yg[:, m] & em5[:, m]).sum() / max(1, yg[:, m].sum()))
    r["n"] = int(len(linhas_idx))
    porpedido = np.stack([h[:, 0], h[:, :3].any(1), hs[:, 0], hs.any(1)], 1).astype(int)
    return r, porpedido

## Treino e avaliação

In [ ]:
resultados = {"receita": {"epocas": EPOCAS, "lam": LAM, "lote": LOTE, "lr": LR, "semente": SEMENTE},
              "referencia_gemma2": universo["referencia_gemma2"]["teste"], "vetores": {}}
porpedido_teste = {}
for nome, pasta in VETORES.items():
    t0 = time.time()
    if isinstance(pasta, (tuple, list)):
        partes = [carregar_vetores(p) for p in pasta]
        X = np.mean([x for x, _ in partes], axis=0); X /= np.linalg.norm(X, axis=1, keepdims=True)
        meta = {"composicao": "media", "partes": [m for _, m in partes], "model": partes[0][1].get("model")}
        del partes
    else:
        X, meta = carregar_vetores(pasta)
    print(f"[{nome}] {X.shape} | modelo {meta.get('model')} | {meta.get('composicao') or 'max_tokens ' + str(meta.get('max_tokens'))}", flush=True)
    Wg, bg = treinar(X[IDX["train"]], YG[IDX["train"]]); print(f"  grupo treinado ({time.time() - t0:.0f}s)", flush=True)
    Ws, bs = treinar(X[IDX["train"]], YS[IDX["train"]]); print(f"  subclasse treinada ({time.time() - t0:.0f}s)", flush=True)
    np.savez_compressed(SAIDA / f"modelo_{nome}.npz", grupo_W=Wg, grupo_b=bg, subclasse_W=Ws, subclasse_b=bs,
                        grupos=np.array(GRUPOS), subclasses=np.array(SUBCLASSES), meta_json=np.array(json.dumps(meta)))
    r = {"meta": meta, "dimensao": int(X.shape[1])}
    for parte in ("validation", "test"):
        og, os_ = decodificar(X[IDX[parte]], Wg, bg, Ws, bs)
        r[parte], pp = metricas(og, os_, IDX[parte])
        if parte == "test":
            porpedido_teste[nome] = pp
            # pedidos longos (texto TAC acima de LIMITE_LONGOS tokens): o mesmo grupo para todas as variantes
            tr = TOKENS[IDX["test"]] > LIMITE_LONGOS
            if True:
                for rot, m in (("teste_longos", tr), ("teste_curtos", ~tr)):
                    r[rot], _ = metricas(og[m], os_[m], IDX["test"][m])
    resultados["vetores"][nome] = r
    print(f"  teste: grupo Hit@1={r['test']['grupo_hit@1']:.3f} Hit@3={r['test']['grupo_hit@3']:.3f} "
          f"subclasse Hit@1={r['test']['subclasse_hit@1']:.3f} Hit@3={r['test']['subclasse_hit@3']:.3f} "
          f"raros@5={r['test']['recall@5_raros']:.3f} ({time.time() - t0:.0f}s)", flush=True)

## Comparação com o Gemma 2 (mesmo teste, pedido a pedido)

In [ ]:
ref = universo["referencia_gemma2"]
ordem = [IDS[i] for i in IDX["test"]]
B = np.array([ref["por_pedido_teste"][n] for n in ordem])
rng = np.random.default_rng(0); n = len(B)
amostras = [rng.integers(0, n, n) for _ in range(N_BOOTSTRAP)]
print("referência Gemma 2 TAC:", {k: round(v, 3) for k, v in ref["teste"].items()})
for nome, A in porpedido_teste.items():
    comp = {}
    for j, col in enumerate(ref["colunas_por_pedido"]):
        d = A[:, j].astype(float) - B[:, j]
        boot = np.array([d[s].mean() for s in amostras]); lo, hi = np.percentile(boot, [2.5, 97.5])
        comp[col] = {"dif": round(float(d.mean()), 4), "ic95": [round(float(lo), 4), round(float(hi), 4)],
                     "melhora": int((d > 0).sum()), "piora": int((d < 0).sum())}
        print(f"{nome} - Gemma | {col:12s} {d.mean() * 100:+.1f} pts  IC95% [{lo * 100:+.1f}; {hi * 100:+.1f}]  "
              f"melhora {comp[col]['melhora']}, piora {comp[col]['piora']}")
    resultados["vetores"][nome]["comparacao_gemma2_teste"] = comp
    for rot in ("teste_curtos", "teste_longos"):
        if rot in resultados["vetores"][nome]:
            x = resultados["vetores"][nome][rot]
            print(f"  {rot:20s} n={x['n']:4d} grupo Hit@1={x['grupo_hit@1']:.3f} subclasse Hit@1={x['subclasse_hit@1']:.3f}")
json.dump(resultados, open(SAIDA / "resultados.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("gravado:", SAIDA / "resultados.json")